# Bloco 1 — EDA adversarial

Auditoria das duas fontes de dados do Challenge 002 (Redesign de Suporte).
Cada hipótese é testada para refutar ou comprovar; cada achado vem com
evidência crua. O bloco fecha com um veredito PASS / WARN / FAIL por fonte.

| Dataset | Arquivo | Linhas esperadas |
|---------|---------|------------------|
| D1 | `customer_support_tickets.csv` | 8.469 |
| D2 | `all_tickets_processed_improved_v3.csv` | 47.837 |

> **Nota de parsing:** o D1 tem descrições que ocupam várias linhas físicas
> dentro do CSV (campos entre aspas com quebras de linha). O parsing correto colapsa isso em 8.469 tickets. Uma contagem de ~29.808 indica bug de parsing.

## 1.0 — Setup e carga

Imports, função de carga parametrizada e sanity check dos dois datasets.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

In [2]:
def load_dataset(filename: str) -> pd.DataFrame:
    """Carrega um CSV de `solution/datasets/` e retorna o DataFrame.

    O caminho é resolvido de forma relativa e reproduzível: procura a pasta
    `datasets/` a partir do diretório de execução do notebook (`solution/`)
    e, como fallback, sobe na árvore até encontrar `solution/datasets/` —
    sem nenhum caminho absoluto de máquina.

    Leitura em UTF-8 com `low_memory=False` (arquivo inteiro inferido de uma
    vez), o que evita o DtypeWarning de tipo misto do parser em chunks.
    """
    candidates = [Path("datasets")]
    candidates += [parent / "solution" / "datasets" for parent in [Path.cwd(), *Path.cwd().parents]]
    data_dir = next((d for d in candidates if d.is_dir()), None)
    if data_dir is None:
        raise FileNotFoundError("Pasta solution/datasets/ não encontrada a partir de " + str(Path.cwd()))
    return pd.read_csv(data_dir / filename, encoding="utf-8", low_memory=False)

In [3]:
df1 = load_dataset("customer_support_tickets.csv")
df2 = load_dataset("all_tickets_processed_improved_v3.csv")

In [4]:
# Sanity check — confronta o shape real com o esperado.
EXPECTED = {"D1": 8_469, "D2": 47_837}

for name, df in [("D1", df1), ("D2", df2)]:
    rows = len(df)
    status = "OK" if rows == EXPECTED[name] else "*** AVISO: ESPERADO " + f"{EXPECTED[name]:,}" + " ***"
    print(f"{name}: shape = {df.shape}  [{status}]")

D1: shape = (8469, 17)  [OK]
D2: shape = (47837, 2)  [OK]


In [5]:
# Colunas de cada dataset — referência para as seções de auditoria.
print("D1 —", len(df1.columns), "colunas:")
print(list(df1.columns))
print()
print("D2 —", len(df2.columns), "colunas:")
print(list(df2.columns))

D1 — 17 colunas:
['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']

D2 — 2 colunas:
['Document', 'Topic_group']


## 1.1 — D1: integridade e semântica dos nulos

Investiga se os nulos de `Customer Satisfaction Rating` e `Resolution` são estruturais, presentes apenas em tickets fora do status `Closed`. Saída: tabela de nulos por status.

In [6]:
# Parâmetros do bloco 1.1–1.6. Ajuste aqui.
N_DESC = 8         # amostras de Ticket Description exibidas em 1.2
N_RES = 8          # amostras de Resolution exibidas em 1.2
N_MISC_BASE = 50   # textos de Miscellaneous para dispersão de termos em 1.3
N_MISC_SHOW = 12   # textos de Miscellaneous exibidos em 1.3

import re
from collections import Counter

# Stopwords para as contagens lexicais de 1.3 e 1.4. Lista curta de termos
# funcionais e de saudação. O resíduo que escapa vira evidência de ruído.
STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "for", "on", "with", "is",
    "are", "was", "be", "please", "hi", "hello", "dear", "thanks", "thank",
    "regards", "best", "kind", "you", "your", "we", "it", "this", "that", "at",
    "as", "by", "from", "re", "pm", "am", "not", "no", "if", "my", "me", "our",
    "us",
}


def tokenize(text: str) -> list[str]:
    """Tokens alfabéticos minúsculos, comprimento >= 3, sem stopwords."""
    return [t for t in re.findall(r"[a-z]+", str(text).lower())
            if len(t) >= 3 and t not in STOPWORDS]

In [7]:
OUTCOME_COLS = [
    "Customer Satisfaction Rating", "Resolution",
    "First Response Time", "Time to Resolution",
]

print("Tickets por Ticket Status")
print(df1["Ticket Status"].value_counts().to_string())

print("\nNulos por status nas colunas de desfecho")
null_by_status = (
    df1.groupby("Ticket Status")[OUTCOME_COLS].apply(lambda g: g.isna().sum())
)
print(null_by_status.to_string())

print("\nTicket ID único:", df1["Ticket ID"].is_unique,
      "| distintos:", df1["Ticket ID"].nunique(), "de", len(df1))
print("\ndtypes")
print(df1.dtypes.to_string())

# Coerência estrutural dos nulos.
# Grupo de fechamento: só deveria estar preenchido em Closed.
closing = ["Customer Satisfaction Rating", "Resolution", "Time to Resolution"]
not_closed = df1["Ticket Status"] != "Closed"
viol_closing_filled = int(df1.loc[not_closed, closing].notna().any(axis=1).sum())
viol_closed_null = int(df1.loc[~not_closed, closing].isna().any(axis=1).sum())
# First Response Time: só deveria estar ausente em Open.
is_open = df1["Ticket Status"] == "Open"
viol_frt = int(df1.loc[~is_open, "First Response Time"].isna().sum()) \
    + int(df1.loc[is_open, "First Response Time"].notna().sum())

print("\nViolações de coerência")
print("desfecho de fechamento preenchido fora de Closed:", viol_closing_filled)
print("desfecho de fechamento nulo em Closed:", viol_closed_null)
print("First Response Time fora do padrão 'ausente só em Open':", viol_frt)

nulos_estruturais = (viol_closing_filled == 0
                     and viol_closed_null == 0 and viol_frt == 0)
id_integro = bool(df1["Ticket ID"].is_unique)

Tickets por Ticket Status
Ticket Status
Pending Customer Response    2881
Open                         2819
Closed                       2769

Nulos por status nas colunas de desfecho
                           Customer Satisfaction Rating  Resolution  First Response Time  Time to Resolution
Ticket Status                                                                                               
Closed                                                0           0                    0                   0
Open                                               2819        2819                 2819                2819
Pending Customer Response                          2881        2881                    0                2881

Ticket ID único: True | distintos: 8469 de 8469

dtypes
Ticket ID                         int64
Customer Name                    object
Customer Email                   object
Customer Age                      int64
Customer Gender                  object
Product Purcha

**Achado 1.1.** Os nulos são estruturais. As três colunas de fechamento
(`Customer Satisfaction Rating`, `Resolution`, `Time to Resolution`) estão
preenchidas exatamente nos 2.769 tickets `Closed` e nulas em todos os demais.
`First Response Time` está ausente apenas nos 2.819 tickets `Open`. Zero
violações nos três testes de coerência. `Ticket ID` é único nas 8.469 linhas.

## 1.2 — D1: sinteticidade

Investiga o quanto do texto livre é gerado: % de placeholder `{product_purchased}` cru nas descrições e amostra de `Resolution`, separando texto sintético dos metadados categóricos aproveitáveis.

In [8]:
PLACEHOLDER = "{product_purchased}"
n = len(df1)
has_ph = df1["Ticket Description"].str.contains(PLACEHOLDER, regex=False)
pct_ph = has_ph.sum() / n * 100
print(f"Descrições com placeholder cru '{PLACEHOLDER}': "
      f"{has_ph.sum()} de {n} ({pct_ph:.1f}%)")

print(f"\nAmostra de {N_DESC} Ticket Description")
for i, x in enumerate(df1["Ticket Description"].head(N_DESC), 1):
    print(f"[{i}] {x[:160]!r}")

print(f"\nAmostra de {N_RES} Resolution (não nulos)")
for i, x in enumerate(df1["Resolution"].dropna().head(N_RES), 1):
    print(f"[{i}] {x!r}")

print("\nMetadados categóricos")
for c in ["Ticket Type", "Ticket Channel", "Ticket Priority", "Product Purchased"]:
    print(f"\n{c}: {df1[c].nunique()} categorias")
    print(df1[c].value_counts().head(10).to_string())

import math
print("\nUniformidade das categorias operacionais (entropia normalizada H/Hmax)")
uniformidade = {}
for c in ["Ticket Type", "Ticket Channel", "Ticket Priority"]:
    p = df1[c].value_counts(normalize=True)
    h = -(p * p.map(math.log2)).sum()
    uniformidade[c] = h / math.log2(len(p))
    print(f"  {c}: {uniformidade[c]:.4f} ({len(p)} categorias)")
uniformidade_min = min(uniformidade.values())

texto_livre_sintetico = pct_ph == 100.0

Descrições com placeholder cru '{product_purchased}': 8469 de 8469 (100.0%)

Amostra de 8 Ticket Description
[1] "I'm having an issue with the {product_purchased}. Please assist.\n\nYour billing zip code is: 71701.\n\nWe appreciate that you have requested a website address.\n\nPl"
[2] "I'm having an issue with the {product_purchased}. Please assist.\n\nIf you need to change an existing product.\n\nI'm having an issue with the {product_purchased}. "
[3] "I'm facing a problem with my {product_purchased}. The {product_purchased} is not turning on. It was working fine until yesterday, but now it doesn't respond.\n\n1"
[4] "I'm having an issue with the {product_purchased}. Please assist.\n\nIf you have a problem you're interested in and I'd love to see this happen, please check out t"
[5] "I'm having an issue with the {product_purchased}. Please assist.\n\n\nNote: The seller is not responsible for any damages arising out of the delivery of the battle"
[6] "I'm facing a problem with my {prod

**Achado 1.2.** O texto livre é sintético. 100% das 8.469 descrições contêm o
placeholder cru `{product_purchased}`, sem substituição pelo produto real. Os
valores de `Resolution` são sentenças aleatórias sem relação com o chamado. Os
metadados categóricos têm domínio fechado: `Ticket Type` (5), `Ticket Channel`
(4), `Ticket Priority` (4), `Product Purchased` (42). A entropia normalizada de
`Ticket Type`, `Ticket Channel` e `Ticket Priority` fica acima de 0,999.
Distribuição próxima da uniforme indica geração aleatória. Os metadados têm
formato íntegro e sinal de negócio ausente.

## 1.3 — D2: distribuição de classes

Investiga o balanceamento das 8 categorias (top-3 ≈ 66%) e testa se `Miscellaneous` é triagem preguiçosa ou categoria legítima, amostrando seus textos.

In [9]:
print("Distribuição de Topic_group")
vc = df2["Topic_group"].value_counts()
dist = pd.DataFrame({"contagem": vc, "pct": (vc / len(df2) * 100).round(2)})
print(dist.to_string())
top3_pct = float(dist["pct"].head(3).sum())
misc_pct = float(dist.loc["Miscellaneous", "pct"])
print(f"top-3 acumulado: {top3_pct:.2f}%")

# Dispersão de termos em Miscellaneous sobre N_MISC_BASE textos espaçados.
misc = df2.loc[df2["Topic_group"] == "Miscellaneous", "Document"].reset_index(drop=True)
step_base = max(1, len(misc) // N_MISC_BASE)
misc_base = misc.iloc[::step_base].head(N_MISC_BASE)
freq_doc = Counter()
for t in misc_base:
    freq_doc.update(set(tokenize(t)))   # frequência por documento
print(f"\nTermos mais frequentes em {len(misc_base)} textos de Miscellaneous "
      f"(nº de documentos que contêm o termo)")
for term, c in freq_doc.most_common(15):
    print(f"{term:>14}: {c}")
print(f"termos distintos nos {len(misc_base)} textos: {len(freq_doc)}")

print(f"\nAmostra de {N_MISC_SHOW} textos de Miscellaneous")
step_show = max(1, len(misc) // N_MISC_SHOW)
for i, x in enumerate(misc.iloc[::step_show].head(N_MISC_SHOW), 1):
    print(f"[{i}] {x[:150]!r}")

Distribuição de Topic_group
                       contagem    pct
Topic_group                           
Hardware                  13617  28.47
HR Support                10915  22.82
Access                     7125  14.89
Miscellaneous              7060  14.76
Storage                    2777   5.81
Purchase                   2464   5.15
Internal Project           2119   4.43
Administrative rights      1760   3.68
top-3 acumulado: 66.18%

Termos mais frequentes em 50 textos de Miscellaneous (nº de documentos que contêm o termo)
        change: 15
     wednesday: 12
           add: 11
      thursday: 11
       analyst: 9
      attached: 8
          name: 8
          sent: 8
       tuesday: 7
         owner: 7
        oracle: 6
           let: 6
         these: 6
           ext: 6
          have: 6
termos distintos nos 50 textos: 520

Amostra de 12 textos de Miscellaneous
[1] 'mail verification warning hi has got attached please addresses best regards monitoring analyst verification warn

**Achado 1.3.** As classes são desbalanceadas. As oito categorias vão de 28,47%
(`Hardware`) a 3,68% (`Administrative rights`); o top-3 concentra 66,18%.
`Miscellaneous` é o quarto grupo com 14,76% e funciona como depósito difuso: nos
textos amostrados os termos mais frequentes aparecem em poucos documentos e
cobrem assuntos sem tema comum (troca de gestor, verificação de e-mail,
recuperação de impressora, clonagem de dashboard).

## 1.4 — D2: sinal do texto pré-processado

Investiga se o texto processado ainda carrega sinal classificável: comprimento, vocabulário e ruído residual do pré-processamento.

In [10]:
lens_char = df2["Document"].str.len()
lens_tok = df2["Document"].map(lambda s: len(str(s).split()))
print("Comprimento de Document")
print(f"  caracteres: media {lens_char.mean():.1f} | mediana {lens_char.median():.0f} "
      f"| de {lens_char.min()} a {lens_char.max()}")
print(f"  tokens:     media {lens_tok.mean():.1f} | mediana {lens_tok.median():.0f} "
      f"| de {lens_tok.min()} a {lens_tok.max()}")

vocab = Counter()
for t in df2["Document"]:
    vocab.update(tokenize(t))
n_vocab = len(vocab)
hapax = sum(1 for w, c in vocab.items() if c == 1)
print(f"\nVocabulario: {n_vocab} termos | hapax (freq 1): {hapax} "
      f"({hapax / n_vocab * 100:.1f}%)")

print("\nTop 8 termos por Topic_group")
tops = {}
for g, sub in df2.groupby("Topic_group"):
    c = Counter()
    for t in sub["Document"]:
        c.update(tokenize(t))
    tops[g] = [w for w, _ in c.most_common(8)]
    print(f"{g:>22}: {tops[g]}")

# Termos genéricos: presentes no top-8 de metade ou mais das classes.
appear = Counter()
for terms in tops.values():
    appear.update(terms)
half = len(tops) / 2
shared = sorted(w for w, k in appear.items() if k >= half)
print(f"\nTermos genéricos (top-8 de >= {int(half)} das {len(tops)} classes): {shared}")

Comprimento de Document
  caracteres: media 291.9 | mediana 175 | de 7 a 7015
  tokens:     media 43.6 | mediana 26 | de 2 a 981



Vocabulario: 12177 termos | hapax (freq 1): 2714 (22.3%)

Top 8 termos por Topic_group
                Access: ['confluence', 'card', 'user', 'access', 'password', 'license', 'create', 'users']
 Administrative rights: ['upgrade', 'update', 'issues', 'sent', 'have', 'can', 'issue', 'version']


            HR Support: ['leaver', 'error', 'form', 'starter', 'leave', 'did', 'access', 'sent']


              Hardware: ['sent', 'access', 'have', 'can', 'issue', 'help', 'tuesday', 'wednesday']
      Internal Project: ['code', 'setup', 'project', 'codes', 'attached', 'form', 'pipeline', 'new']
         Miscellaneous: ['change', 'add', 'name', 'sent', 'approval', 'tuesday', 'wednesday', 'assigned']
              Purchase: ['purchase', 'administrator', 'log', 'order', 'receive', 'purchased', 'item', 'new']


               Storage: ['mailbox', 'folder', 'access', 'wants', 'shared', 'accept', 'decline', 'size']

Termos genéricos (top-8 de >= 4 das 8 classes): ['access', 'sent']


**Achado 1.4.** Há sinal com ruído. Os documentos têm mediana de 26 tokens e
vocabulário de 12.177 termos, dos quais 22,3% são hapax. Cada classe tem
vocabulário próprio (`confluence` e `password` em `Access`, `mailbox` e `folder`
em `Storage`, `purchase` e `order` em `Purchase`, `leaver` e `starter` em
`HR Support`). Termos genéricos aparecem no top-8 de quatro ou mais classes
(`access`, `sent`), e sobram tokens de dia da semana e de função (`tuesday`,
`wednesday`, `have`, `can`), resíduo de pré-processamento incompleto. O texto é
classificável e ainda pede limpeza.

## 1.5 — Cruzabilidade D1 × D2

Investiga se existe ponte real entre os datasets: comparação de esquemas, taxonomias e chaves candidatas.

In [11]:
cols1, cols2 = set(df1.columns), set(df2.columns)
print("Colunas em comum entre D1 e D2:", (cols1 & cols2) or "nenhuma")
print(f"D1: {len(cols1)} colunas | D2: {len(cols2)} colunas {sorted(cols2)}")

tt = set(df1["Ticket Type"].unique())
tg = set(df2["Topic_group"].unique())
print("\nTaxonomia D1 (Ticket Type):", sorted(tt))
print("Taxonomia D2 (Topic_group):", sorted(tg))
inter_raw = tt & tg
inter_lower = {x.lower() for x in tt} & {x.lower() for x in tg}
print("Sobreposição de categorias (exata):", inter_raw or "nenhuma")
print("Sobreposição de categorias (minúsculas):", inter_lower or "nenhuma")

print("\nTicket ID de D1:", df1["Ticket ID"].min(), "a", df1["Ticket ID"].max(),
      "(sequencial interno)")
print("D2 não possui coluna de identificador.")

cruzavel = bool(cols1 & cols2) or bool(inter_lower)

Colunas em comum entre D1 e D2: nenhuma
D1: 17 colunas | D2: 2 colunas ['Document', 'Topic_group']

Taxonomia D1 (Ticket Type): ['Billing inquiry', 'Cancellation request', 'Product inquiry', 'Refund request', 'Technical issue']
Taxonomia D2 (Topic_group): ['Access', 'Administrative rights', 'HR Support', 'Hardware', 'Internal Project', 'Miscellaneous', 'Purchase', 'Storage']
Sobreposição de categorias (exata): nenhuma
Sobreposição de categorias (minúsculas): nenhuma

Ticket ID de D1: 1 a 8469 (sequencial interno)
D2 não possui coluna de identificador.


**Achado 1.5.** Não há ponte entre as fontes. Nenhuma coluna em comum (D1 tem 17,
D2 tem 2). As taxonomias não se sobrepõem: `Ticket Type` classifica intenção
(`Refund request`, `Technical issue`, `Billing inquiry`); `Topic_group`
classifica área técnica (`Hardware`, `Access`, `Storage`). D2 não tem
identificador e o `Ticket ID` de D1 é sequencial interno sem contraparte. As
fontes são independentes.

## 1.6 — Veredito por fonte

Consolida a auditoria numa tabela fonte × dimensão com PASS / WARN / FAIL e uma linha de justificativa por célula.

In [12]:
def status(cond, ok="PASS", bad="FAIL"):
    return ok if cond else bad

verdict = {
    "D1": {
        "integridade estrutural": {
            "status": status(id_integro),
            "justificativa": "Ticket ID único nas 8.469 linhas e dtypes consistentes.",
        },
        "semântica dos nulos": {
            "status": status(nulos_estruturais),
            "justificativa": "Nulos 100% estruturais por Ticket Status, zero violações de coerência.",
        },
        "texto livre": {
            "status": "FAIL" if texto_livre_sintetico else "WARN",
            "justificativa": "100% das descrições com placeholder cru e Resolution aleatória.",
        },
        "metadados categóricos": {
            "status": "WARN",
            "justificativa": f"Categorias válidas e sem nulos, entropia normalizada >= {uniformidade_min:.4f} indica distribuição uniforme sem sinal de negócio.",
        },
    },
    "D2": {
        "rótulos": {
            "status": "WARN",
            "justificativa": f"Oito classes desbalanceadas (top-3 {top3_pct:.0f}%) e Miscellaneous difuso ({misc_pct:.0f}%).",
        },
        "sinal do texto": {
            "status": "WARN",
            "justificativa": "Vocabulário discriminativo por classe com ruído residual de pré-processamento.",
        },
    },
    "D1 × D2": {
        "cruzabilidade": {
            "status": "FAIL" if not cruzavel else "WARN",
            "justificativa": "Sem coluna, taxonomia ou identificador em comum entre as fontes.",
        },
    },
}

import json
print(json.dumps(verdict, ensure_ascii=False, indent=2))

{
  "D1": {
    "integridade estrutural": {
      "status": "PASS",
      "justificativa": "Ticket ID único nas 8.469 linhas e dtypes consistentes."
    },
    "semântica dos nulos": {
      "status": "PASS",
      "justificativa": "Nulos 100% estruturais por Ticket Status, zero violações de coerência."
    },
    "texto livre": {
      "status": "FAIL",
      "justificativa": "100% das descrições com placeholder cru e Resolution aleatória."
    },
    "metadados categóricos": {
      "status": "WARN",
      "justificativa": "Categorias válidas e sem nulos, entropia normalizada >= 0.9997 indica distribuição uniforme sem sinal de negócio."
    }
  },
  "D2": {
    "rótulos": {
      "status": "WARN",
      "justificativa": "Oito classes desbalanceadas (top-3 66%) e Miscellaneous difuso (15%)."
    },
    "sinal do texto": {
      "status": "WARN",
      "justificativa": "Vocabulário discriminativo por classe com ruído residual de pré-processamento."
    }
  },
  "D1 × D2": {
    "cruzab

In [13]:
from IPython.display import Markdown, display

rows = ["| Fonte | Dimensão | Status | Justificativa |",
        "|-------|----------|--------|---------------|"]
for fonte, dims in verdict.items():
    for dim, cell in dims.items():
        rows.append(f"| {fonte} | {dim} | {cell['status']} | {cell['justificativa']} |")
md_table = "\n".join(rows)
print(md_table)
display(Markdown(md_table))

| Fonte | Dimensão | Status | Justificativa |
|-------|----------|--------|---------------|
| D1 | integridade estrutural | PASS | Ticket ID único nas 8.469 linhas e dtypes consistentes. |
| D1 | semântica dos nulos | PASS | Nulos 100% estruturais por Ticket Status, zero violações de coerência. |
| D1 | texto livre | FAIL | 100% das descrições com placeholder cru e Resolution aleatória. |
| D1 | metadados categóricos | WARN | Categorias válidas e sem nulos, entropia normalizada >= 0.9997 indica distribuição uniforme sem sinal de negócio. |
| D2 | rótulos | WARN | Oito classes desbalanceadas (top-3 66%) e Miscellaneous difuso (15%). |
| D2 | sinal do texto | WARN | Vocabulário discriminativo por classe com ruído residual de pré-processamento. |
| D1 × D2 | cruzabilidade | FAIL | Sem coluna, taxonomia ou identificador em comum entre as fontes. |


| Fonte | Dimensão | Status | Justificativa |
|-------|----------|--------|---------------|
| D1 | integridade estrutural | PASS | Ticket ID único nas 8.469 linhas e dtypes consistentes. |
| D1 | semântica dos nulos | PASS | Nulos 100% estruturais por Ticket Status, zero violações de coerência. |
| D1 | texto livre | FAIL | 100% das descrições com placeholder cru e Resolution aleatória. |
| D1 | metadados categóricos | WARN | Categorias válidas e sem nulos, entropia normalizada >= 0.9997 indica distribuição uniforme sem sinal de negócio. |
| D2 | rótulos | WARN | Oito classes desbalanceadas (top-3 66%) e Miscellaneous difuso (15%). |
| D2 | sinal do texto | WARN | Vocabulário discriminativo por classe com ruído residual de pré-processamento. |
| D1 × D2 | cruzabilidade | FAIL | Sem coluna, taxonomia ou identificador em comum entre as fontes. |